In [ ]:
input_tif = '/media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/YZ 2689.tif'

output_tif = '/media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/sample_destriped.tif'

In [1]:
import sys
# Ensure local package modules are on path
sys.path.insert(0, '/home/home/Documents/UNet/aind-smartspim-destripe')
sys.path.insert(0, '/home/home/Documents/UNet/aind-smartspim-destripe/code/aind_smartspim_destripe')

import os
import zarr
import tifffile
import numpy as np
from aind_smartspim_destripe.filtering import log_space_fft_filtering

# 1. Read the TIFF into a NumPy array
tif_path = '/media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/B3GT084_BB1_170722-a_-1-128_3D_OVERLAY-YZ.tif'
volume = tifffile.imread(tif_path)
print(f"Loaded volume with shape {volume.shape} and dtype {volume.dtype}")

# Handle single-slice case: add batch dimension if needed
if volume.ndim == 2:
    volume = volume[np.newaxis, ...]

# 2. (Optional) Write to Zarr for reference
zarr_path = '/media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/YZ_2689.zarr'
chunks = (1,) + volume.shape[1:]
root = zarr.open_group(zarr_path, mode='w')
root.create_array('data', shape=volume.shape, dtype=volume.dtype, chunks=chunks, overwrite=True)
root['data'][:] = volume
print(f"Wrote volume to Zarr at {zarr_path}")

# 3. Apply authors' sample parameters for their original dataset
# Define output directory for results
results_dir = '/media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/destripe_results_authors'
os.makedirs(results_dir, exist_ok=True)

# No-cells configuration
# No-cells configuration
configurations = {
    'no_cells': {'sigma': 128, 'max_threshold': 12},
    'cells': {'sigma': 64, 'max_threshold': 3}
}

for cfg_name, cfg in configurations.items():
    sigma = cfg['sigma']
    thr = cfg['max_threshold']
    print(f"Processing configuration: {cfg_name} (σ={sigma}, threshold={thr})")
    destriped_slices = []
    for frame in volume:
        # For vertical stripes, transpose image to treat stripes as horizontal
        frame_proc = frame.T
        out_proc = log_space_fft_filtering(
            frame_proc.astype(np.float64),
            wavelet='db3',          # Daubechies-3 wavelet
            level=None,             # full decomposition
            sigma=sigma,
            max_threshold=thr
        )
        # Transpose back to original orientation
        out = out_proc.T
        destriped_slices.append(out)
    destriped_vol = np.stack(destriped_slices, axis=0)

    # Save result as multi-page TIFF
    output_fname = f"destriped_{cfg_name}.tif"
    output_path = os.path.join(results_dir, output_fname)
    with tifffile.TiffWriter(output_path) as tifw:
        if destriped_vol.ndim == 3 and destriped_vol.shape[0] == 1:
            img = destriped_vol[0]
            tifw.write(img.astype(np.float32))
        else:
            for slc in destriped_vol:
                tifw.write(slc.astype(np.float32))
    print(f"Saved: {output_path}")

print("Destriping with authors' parameters complete. Results in:", results_dir)

Loaded volume with shape (2667, 1345, 128) and dtype uint8


KeyboardInterrupt: 

In [7]:
import sys
# Ensure local package modules are on path
sys.path.insert(0, '/home/home/Documents/UNet/aind-smartspim-destripe')
sys.path.insert(0, '/home/home/Documents/UNet/aind-smartspim-destripe/code/aind_smartspim_destripe')

import os
import zarr
import tifffile
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from aind_smartspim_destripe.filtering import log_space_fft_filtering
import multiprocessing

# 1. Read the TIFF into a NumPy array
tif_path = '/media/home/DATA10TB/MITOCHONDRIA/3D_SANITY_SEG/destripe/b3GT073_bl1_161015-a-YZ.tif'
volume = tifffile.imread(tif_path)
print(f"Loaded volume with shape {volume.shape} and dtype {volume.dtype}")

# Handle single-slice case: add batch dimension if needed
if volume.ndim == 2:
    volume = volume[np.newaxis, ...]

# 2. (Optional) Write to Zarr for reference
zarr_path = '/media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/YZ_2689.zarr'
chunks = (1,) + volume.shape[1:]
root = zarr.open_group(zarr_path, mode='w')
root.create_array('data', shape=volume.shape, dtype=volume.dtype, chunks=chunks, overwrite=True)
root['data'][:] = volume
print(f"Wrote volume to Zarr at {zarr_path}")

# 3. Define helper for multiprocessing

def process_frame(frame, sigma, thr):
    # Transpose to treat vertical stripes as horizontal
    frame_proc = frame.T
    # Apply log-space FFT filtering
    out_proc = log_space_fft_filtering(
        frame_proc.astype(np.float64),
        wavelet='db3',          # Daubechies-3 wavelet
        level=None,             # full decomposition
        sigma=sigma,
        max_threshold=thr
    )
    # Transpose back to original orientation
    return out_proc.T

# 4. Apply authors' sample parameters with multiprocessing
results_dir = '/media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/destripe_results_authors'
os.makedirs(results_dir, exist_ok=True)

configurations = {
    'no_cells': {'sigma': 128, 'max_threshold': 12},
    'cells': {'sigma': 64, 'max_threshold': 3}
}

n_workers = min(multiprocessing.cpu_count(), len(volume))
print(f"Using {n_workers} parallel workers")

for cfg_name, cfg in configurations.items():
    sigma = cfg['sigma']
    thr = cfg['max_threshold']
    print(f"Processing configuration: {cfg_name} (σ={sigma}, threshold={thr})")
    # Use ProcessPoolExecutor to parallelize frame processing
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        # Submit tasks
        futures = [executor.submit(process_frame, frame, sigma, thr) for frame in volume]
        # Retrieve results
        destriped_slices = [f.result() for f in futures]
    destriped_vol = np.stack(destriped_slices, axis=0)

    # Save result as multi-page TIFF
    output_fname = f"destriped_{cfg_name}.tif"
    output_path = os.path.join(results_dir, output_fname)
    with tifffile.TiffWriter(output_path, bigtiff=True) as tifw:
        if destriped_vol.ndim == 3 and destriped_vol.shape[0] == 1:
            img = destriped_vol[0]
            tifw.write(img.astype(np.float32))
        else:
            for slc in destriped_vol:
                tifw.write(slc.astype(np.float32))
    print(f"Saved: {output_path}")

print("Destriping with authors' parameters complete. Results in:", results_dir)


Loaded volume with shape (2660, 1021, 541) and dtype uint8
Wrote volume to Zarr at /media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/YZ_2689.zarr
Using 24 parallel workers
Processing configuration: no_cells (σ=128, threshold=12)
Saved: /media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/destripe_results_authors/destriped_no_cells.tif
Processing configuration: cells (σ=64, threshold=3)
Saved: /media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/destripe_results_authors/destriped_cells.tif
Destriping with authors' parameters complete. Results in: /media/home/DATA10TB/MITOCHONDRIA/GT-REDO/raw/test/destripe_results_authors
